# Support Vector Machine (SVM) – 30‑Minute Waste Forecast

This notebook trains an SVM regressor for each canteen section. Features are standardised. The model is trained directly on the waste amount (including zeros).

Saves models and plots to `deployment_models/`.

In [ ]:
# 1. Imports
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib
from google.colab import drive

warnings.filterwarnings('ignore')
np.random.seed(42)

# 2. Mount drive & set working directory
drive.mount('/content/drive', force_remount=True)
try:
    os.chdir('/content/drive/MyDrive/UAB/FDS/campus-waste-intelligence')
    print('Directory changed to project folder')
except OSError:
    print("Error: Could not change directory. Please check the path.")

In [ ]:
# 3. Configuration
DATA_PATH   = 'data/food_waste_augmented.csv'
MODEL_DIR   = 'deployment_models'
MODEL_NAME  = 'SVM'

FREQ        = '30min'
ALIGN_SECTIONS = 'union'
TRAIN_RATIO = 0.7
VAL_RATIO   = 0.1
TEST_RATIO  = 0.2
LOOKBACK    = 24
MIN_SAMPLES = 10

os.makedirs(MODEL_DIR, exist_ok=True)

print(f"Aggregation frequency: {FREQ}")
print(f"Model: {MODEL_NAME}")

In [ ]:
# 4. Load and aggregate data (same as before)
df = pd.read_csv(DATA_PATH, parse_dates=['Date'])
daily_section = (
    df.set_index('Date')
      .groupby([pd.Grouper(freq=FREQ), 'Canteen_Section'])['Waste_Weight_kg']
      .sum()
      .reset_index()
      .rename(columns={'Waste_Weight_kg': 'Total_Waste_kg'})
)
daily_wide = (
    daily_section
    .pivot(index='Date', columns='Canteen_Section', values='Total_Waste_kg')
    .sort_index()
)

if ALIGN_SECTIONS == 'common':
    daily_wide = daily_wide.dropna()
else:
    daily_wide = daily_wide.fillna(0)

sections = daily_wide.columns.tolist()
print(f"Sections: {sections}")

In [ ]:
# 5. Feature engineering (same as XGBoost)
def create_features_for_series(series: pd.Series) -> pd.DataFrame:
    df_ml = pd.DataFrame(index=series.index)
    df_ml['y'] = series.values
    df_ml['hour']      = df_ml.index.hour
    df_ml['minute']    = df_ml.index.minute
    df_ml['dayofweek'] = df_ml.index.dayofweek
    df_ml['day']       = df_ml.index.day
    df_ml['month']     = df_ml.index.month
    df_ml['quarter']   = df_ml.index.quarter
    df_ml['weekend']   = (df_ml.index.dayofweek >= 5).astype(int)

    df_ml['hour_sin'] = np.sin(2 * np.pi * df_ml['hour'] / 24)
    df_ml['hour_cos'] = np.cos(2 * np.pi * df_ml['hour'] / 24)
    df_ml['dow_sin'] = np.sin(2 * np.pi * df_ml['dayofweek'] / 7)
    df_ml['dow_cos'] = np.cos(2 * np.pi * df_ml['dayofweek'] / 7)

    for lag in [1, 2, 3, 48, 96]:
        df_ml[f'lag_{lag}'] = df_ml['y'].shift(lag)

    shifted = df_ml['y'].shift(1)
    df_ml['rolling_mean_48'] = shifted.rolling(48).mean()
    df_ml['rolling_std_48']  = shifted.rolling(48).std()
    df_ml['rolling_min_48']  = shifted.rolling(48).min()
    df_ml['rolling_max_48']  = shifted.rolling(48).max()
    df_ml['ewm_mean_48']     = shifted.ewm(span=48).mean()

    df_ml.dropna(inplace=True)
    return df_ml

feature_dfs = {}
for sec in sections:
    feat_df = create_features_for_series(daily_wide[sec])
    feature_dfs[sec] = feat_df

# Align all sections (union)
all_idx = pd.DatetimeIndex([])
for df in feature_dfs.values():
    all_idx = all_idx.union(df.index)
all_idx = all_idx.sort_values()
for sec in sections:
    feature_dfs[sec] = feature_dfs[sec].reindex(all_idx).fillna(0)

print(f"Total periods after alignment: {len(all_idx)}")

In [ ]:
# 6. Split (chronological)
ref_index = feature_dfs[sections[0]].index
n_total   = len(ref_index)
n_test = max(1, int(n_total * TEST_RATIO))
n_val  = max(1, int(n_total * VAL_RATIO))
n_train = n_total - n_val - n_test

train_indices = ref_index[:n_train]
val_indices   = ref_index[n_train:n_train + n_val]
test_indices  = ref_index[n_train + n_val:]

train_mask = ref_index.isin(train_indices)
val_mask   = ref_index.isin(val_indices)
test_mask  = ref_index.isin(test_indices)

print(f"Train: {len(train_indices)}, Val: {len(val_indices)}, Test: {len(test_indices)}")

In [ ]:
# 7. Helper functions
def compute_regression_metrics(y_true, y_pred):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    mask = y_true != 0
    mape = np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100 if mask.sum() > 0 else np.nan
    r2 = r2_score(y_true, y_pred)
    return {'RMSE': rmse, 'MAE': mae, 'MAPE': mape, 'R2': r2}

def plot_predictions(y_true, y_pred, dates, section, max_points=200, save_path=None):
    if len(dates) > max_points:
        step = len(dates) // max_points
        idx = range(0, len(dates), step)
        dates = dates[idx]; y_true = y_true[idx]; y_pred = y_pred[idx]
    plt.figure(figsize=(12,5))
    plt.plot(dates, y_true, label='Actual', marker='o', markersize=3, linestyle='-', linewidth=1)
    plt.plot(dates, y_pred, label='Predicted', marker='x', markersize=3, linestyle='--', linewidth=1)
    plt.xlabel('Date'); plt.ylabel('Waste (kg)')
    plt.title(f'SVM Predictions vs Actual - Section {section}')
    plt.legend(); plt.grid(True); plt.xticks(rotation=45); plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
# 8. Main training loop
all_metrics = []

for sec in sections:
    print(f"\n{'='*60}\nTraining SVM for section {sec}\n{'='*60}")

    df_ml = feature_dfs[sec]
    X = df_ml.drop('y', axis=1)
    y = df_ml['y']

    X_train = X[train_mask]; y_train = y[train_mask]
    X_test  = X[test_mask];  y_test  = y[test_mask]

    if len(X_train) == 0:
        print(f"  Skipping section {sec} – training set empty.")
        continue

    feature_columns = X.columns.tolist()

    # Scale features
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled  = scaler.transform(X_test)

    # Train SVM
    model = SVR(kernel='rbf', C=100, gamma='scale')
    model.fit(X_train_scaled, y_train)
    test_pred = model.predict(X_test_scaled)

    # Metrics
    metrics = compute_regression_metrics(y_test.values, test_pred)
    print("\n--- Performance ---")
    for k,v in metrics.items(): print(f"  {k}: {v:.4f}")

    all_metrics.append({'Section': sec, 'Model': MODEL_NAME, **metrics})

    # Save artefacts
    artifacts = {
        'section': sec,
        'model_name': MODEL_NAME,
        'model': model,
        'scaler': scaler,
        'feature_columns': feature_columns,
        'lookback': LOOKBACK,
        'train_date_range': [train_indices.min().isoformat(), train_indices.max().isoformat()],
        'val_date_range': [val_indices.min().isoformat(), val_indices.max().isoformat()],
        'test_date_range': [test_indices.min().isoformat(), test_indices.max().isoformat()]
    }
    save_path = f'{MODEL_DIR}/section_{sec}_{MODEL_NAME}.joblib'
    joblib.dump(artifacts, save_path)
    print(f"Saved model to {save_path}")

    # Plot predictions
    plot_predictions(y_test.values, test_pred, test_indices, sec,
                     save_path=f'{MODEL_DIR}/section_{sec}_{MODEL_NAME}_predictions.png')

# Save summary
if all_metrics:
    summary_df = pd.DataFrame(all_metrics)
    summary_df.to_csv(f'{MODEL_DIR}/test_metrics_{MODEL_NAME}.csv', index=False)
    print("\nSummary saved.")
else:
    print("No metrics collected.")